In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel,Field
from ast import operator
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

In [12]:
llm = ChatOllama(
    model="qwen3:4b",
    temperature=0.9
)

In [14]:
llm1 = ChatOllama(
    model="qwen3:4b",
    temperature=0.3
)

In [15]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [16]:
def generate_joke(state: JokeState):
    prompt = f'Generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    
    return {'joke': response}

In [17]:
def generate_explanation(state: JokeState):
    prompt= f'Generate an explanation based on the joke {state["joke"]}'
    response = llm1.invoke(prompt).content
    
    return {'explanation': response}

In [18]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer= checkpointer)

In [19]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': 'Here\'s a clean, punny pizza joke that\'s short, relatable, and works for any audience:\n\n**Why did the pizza delivery guy get fired?**  \n*Because he kept saying, "I\'m not a pizza place!"* 😄\n\n*(Punchline explanation: He\'s clearly a *person* delivering pizza, but he accidentally says "I\'m not a pizza place" instead of "I\'m a pizza *delivery* guy" — a classic mix-up with the word "place"!)*\n\nPerfect for sharing on social media, dinner parties, or when you\'re hungry and need a laugh. 🍕',
 'explanation': 'Here\'s a clean, punchy explanation that keeps the joke\'s spirit while making the wordplay crystal clear—perfect for sharing with anyone (kids, adults, non-native speakers, or just hungry folks!):\n\n---\n\n**Why this joke works (and why it’s relatable):**  \nThis pun is a *simple slip of the tongue* gone viral! The delivery guy meant to say **"I’m a pizza *delivery* guy"** (the person who brings pizza) but accidentally said **"I’m not a pizza *pla

In [20]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Here\'s a clean, punny pizza joke that\'s short, relatable, and works for any audience:\n\n**Why did the pizza delivery guy get fired?**  \n*Because he kept saying, "I\'m not a pizza place!"* 😄\n\n*(Punchline explanation: He\'s clearly a *person* delivering pizza, but he accidentally says "I\'m not a pizza place" instead of "I\'m a pizza *delivery* guy" — a classic mix-up with the word "place"!)*\n\nPerfect for sharing on social media, dinner parties, or when you\'re hungry and need a laugh. 🍕', 'explanation': 'Here\'s a clean, punchy explanation that keeps the joke\'s spirit while making the wordplay crystal clear—perfect for sharing with anyone (kids, adults, non-native speakers, or just hungry folks!):\n\n---\n\n**Why this joke works (and why it’s relatable):**  \nThis pun is a *simple slip of the tongue* gone viral! The delivery guy meant to say **"I’m a pizza *delivery* guy"** (the person who brings pizza) but accidentally said **"I

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Here\'s a clean, punny pizza joke that\'s short, relatable, and works for any audience:\n\n**Why did the pizza delivery guy get fired?**  \n*Because he kept saying, "I\'m not a pizza place!"* 😄\n\n*(Punchline explanation: He\'s clearly a *person* delivering pizza, but he accidentally says "I\'m not a pizza place" instead of "I\'m a pizza *delivery* guy" — a classic mix-up with the word "place"!)*\n\nPerfect for sharing on social media, dinner parties, or when you\'re hungry and need a laugh. 🍕', 'explanation': 'Here\'s a clean, punchy explanation that keeps the joke\'s spirit while making the wordplay crystal clear—perfect for sharing with anyone (kids, adults, non-native speakers, or just hungry folks!):\n\n---\n\n**Why this joke works (and why it’s relatable):**  \nThis pun is a *simple slip of the tongue* gone viral! The delivery guy meant to say **"I’m a pizza *delivery* guy"** (the person who brings pizza) but accidentally said **"

### Time Travel

In [23]:
workflow.get_state({'configurable':{"thread_id":"1", "checkpoint_id":'1f1b16ae-9e16-6a4c-8000-51ecc7a0abd2'}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b16ae-9e16-6a4c-8000-51ecc7a0abd2'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-16T01:06:57.370977+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b16ae-9e0c-62fe-bfff-56a03b67a540'}}, tasks=(PregelTask(id='9d3e3a0f-28fa-cbf5-4976-14df0c595365', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Here\'s a clean, punny pizza joke that\'s short, relatable, and works for any audience:\n\n**Why did the pizza delivery guy get fired?**  \n*Because he kept saying, "I\'m not a pizza place!"* 😄\n\n*(Punchline explanation: He\'s clearly a *person* delivering pizza, but he accidentally says "I\'m not a pizza place" instead of "I\'m a pizza *delivery* guy" — a classic mix-up with the word "place"!)*\n\nPerfect for shari

In [24]:
workflow.invoke(None, {'configurable':{"thread_id":"1", "checkpoint_id":'1f1b16ae-9e16-6a4c-8000-51ecc7a0abd2'}} )

{'topic': 'pizza',
 'joke': 'Here\'s a fresh, punny pizza joke for you:\n\n> **Why did the pizza box get fired from the delivery company?**  \n> *Because it was too flimsy to hold the pizza inside... and also kept saying, "I\'m not a pizza, I\'m a crust!"*\n\n*(The joke plays on pizza boxes being fragile + the double meaning of "crust" as both a pizza part and a humble, "not the whole thing" attitude.)* 😄\n\nLet me know if you\'d like a shorter version or a different style!',
 'explanation': 'Here\'s a clear, step-by-step explanation of **why this pizza joke works**—broken down for maximum clarity (with a touch of humor to match your style!). I’ll also include a shorter version and a "for kids" twist at the end, as you requested.\n\n---\n\n### 🔍 **Full Explanation (Why This Joke is Funny)**  \nThis joke is a **classic pun** that plays on two clever layers of meaning:  \n\n1. **The Real-World Problem (Why the box gets fired)**  \n   → Pizza boxes are *inherently fragile* (paper-thin, ea

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Here\'s a fresh, punny pizza joke for you:\n\n> **Why did the pizza box get fired from the delivery company?**  \n> *Because it was too flimsy to hold the pizza inside... and also kept saying, "I\'m not a pizza, I\'m a crust!"*\n\n*(The joke plays on pizza boxes being fragile + the double meaning of "crust" as both a pizza part and a humble, "not the whole thing" attitude.)* 😄\n\nLet me know if you\'d like a shorter version or a different style!', 'explanation': 'Here\'s a clear, step-by-step explanation of **why this pizza joke works**—broken down for maximum clarity (with a touch of humor to match your style!). I’ll also include a shorter version and a "for kids" twist at the end, as you requested.\n\n---\n\n### 🔍 **Full Explanation (Why This Joke is Funny)**  \nThis joke is a **classic pun** that plays on two clever layers of meaning:  \n\n1. **The Real-World Problem (Why the box gets fired)**  \n   → Pizza boxes are *inherently frag

### Updating State

In [27]:
workflow.update_state({"configurable":{"thread_id":"1", 'checkpoint_ns': '', 'checkpoint_id': '1f1b16ae-9e16-6a4c-8000-51ecc7a0abd2'}},{'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b1702-ce40-6494-8001-d56c094e5426'}}

In [28]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1702-ce40-6494-8001-d56c094e5426'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-16T01:44:37.279010+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b16ae-9e16-6a4c-8000-51ecc7a0abd2'}}, tasks=(PregelTask(id='7fc24bf7-94a9-85d9-2f6b-6abccc617bea', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Here\'s a fresh, punny pizza joke for you:\n\n> **Why did the pizza box get fired from the delivery company?**  \n> *Because it was too flimsy to hold the pizza inside... and also kept saying, "I\'m not a pizza, I\'m a crust!"*\n\n*(The joke plays on pizza boxes being fragile + the double meaning of "crust" as both a pizza part 

In [29]:
workflow.invoke(None, {'configurable':{"thread_id":"1", "checkpoint_id":'1f1b1702-ce40-6494-8001-d56c094e5426'}} )

{'topic': 'samosa',
 'joke': 'Here\'s a light, culturally relevant joke for you—**no offensive puns, just food-friendly humor**:\n\n---\n\n**Why did the samosa go to therapy?**  \n*Because it had too many fillings... and the doctor kept saying, "Just one at a time!"* 😄\n\n---\n\n### Why it works:\n- **Play on "fillings"**: Samosas are *packed* with fillings (potatoes, peas, onions, etc.), so "too many fillings" is a pun on both the food and mental stress.\n- **Cultural touch**: No awkward English, no forced slang—just a relatable take on samosa eating that anyone who’s ever struggled with a crispy, stuffed samosa will appreciate.\n- **Universal humor**: Works for anyone who’s ever tried to eat samosas *without* getting overwhelmed by the spices or the grease!\n\nPerfect for sharing with friends, family, or even in a casual food chat. Hope it makes you smile! 🍟',
 'explanation': 'Here\'s a clear, culturally mindful explanation of why this joke works—**without any offensive language, med